<a href="https://colab.research.google.com/github/PardhuVarma55/ELECTRICAL-NETWORK-NATURAL-MODES/blob/main/Frontend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# CELL 16: INSTALL FRONTEND LIBRARY
# ============================================================

!pip -q install gradio

In [2]:
# ============================================================
# CELL 17: FRONTEND CALCULATION ENGINE
# ============================================================

import numpy as np
import matplotlib.pyplot as plt


def calculate_network(
    L1_nH, L2_nH, L3_nH,
    M12_nH, M13_nH, M23_nH,
    C1_pF, C2_pF, C3_pF
):
    """
    Calculates natural frequencies and eigenvectors
    for the 3-coupled-LC network.

    Inputs:
        L, M -> nH
        C   -> pF

    Outputs:
        eigenvalues
        natural frequencies
        eigenvectors
        inductance matrix
        capacitance matrix
    """

    # Convert nH -> H
    L1 = L1_nH * 1e-9
    L2 = L2_nH * 1e-9
    L3 = L3_nH * 1e-9

    M12 = M12_nH * 1e-9
    M13 = M13_nH * 1e-9
    M23 = M23_nH * 1e-9

    # Convert pF -> F
    C1 = C1_pF * 1e-12
    C2 = C2_pF * 1e-12
    C3 = C3_pF * 1e-12

    # Inductance matrix
    L = np.array([
        [L1, M12, M13],
        [M12, L2, M23],
        [M13, M23, L3]
    ], dtype=float)

    # Capacitance matrix
    C = np.diag([C1, C2, C3])

    # Check physical validity
    if not np.allclose(L, L.T):
        raise ValueError("Inductance matrix must be symmetric.")

    if np.any(np.linalg.eigvalsh(L) <= 0):
        raise ValueError(
            "Invalid inductance matrix. "
            "Reduce the mutual inductances."
        )

    # C^-1
    C_inv = np.linalg.inv(C)

    # Eigenvalue problem:
    #
    # C^-1 v = lambda L v
    #
    # L^-1 C^-1 v = lambda v

    A = np.linalg.solve(L, C_inv)

    eigenvalues, eigenvectors = np.linalg.eig(A)

    eigenvalues = np.real_if_close(eigenvalues).real

    # Sort
    order = np.argsort(eigenvalues)

    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]

    # Natural angular frequencies
    omega = np.sqrt(eigenvalues)

    # Natural frequencies
    frequencies = omega / (2 * np.pi)

    # Normalize eigenvectors
    for k in range(3):
        eigenvectors[:, k] /= np.max(
            np.abs(eigenvectors[:, k])
        )

        # Consistent sign
        max_index = np.argmax(
            np.abs(eigenvectors[:, k])
        )

        if eigenvectors[max_index, k] < 0:
            eigenvectors[:, k] *= -1

    return (
        eigenvalues,
        frequencies,
        eigenvectors,
        L,
        C
    )

In [4]:
# ============================================================
# CELL 18: FRONTEND FUNCTION
# ============================================================

def run_frontend(
    L1, L2, L3,
    M12, M13, M23,
    C1, C2, C3
):

    try:

        (
            eigenvalues,
            frequencies,
            eigenvectors,
            L_matrix,
            C_matrix
        ) = calculate_network(
            L1, L2, L3,
            M12, M13, M23,
            C1, C2, C3
        )

        # ----------------------------------------------------
        # Result text
        # ----------------------------------------------------

        result_text = "## ⚡ Natural Modes Result\n\n"

        for k in range(3):

            result_text += (
                f"### Mode {k+1}\n"
                f"- Eigenvalue λ = "
                f"`{eigenvalues[k]:.4e}`\n"
                f"- Angular frequency ω = "
                f"`{np.sqrt(eigenvalues[k]):.4e} rad/s`\n"
                f"- Natural frequency = "
                f"`{frequencies[k]/1e6:.4f} MHz`\n"
                f"- Current distribution = "
                f"`[{eigenvectors[0,k]:+.3f}, "
                f"{eigenvectors[1,k]:+.3f}, "
                f"{eigenvectors[2,k]:+.3f}]`\n\n"
            )

        # ----------------------------------------------------
        # Matrix display
        # ----------------------------------------------------

        matrix_text = (
            "### Inductance Matrix L (nH)\n\n"
            f"```text\n"
            f"{L_matrix / 1e-9}\n"
            f"```\n\n"
            "### Capacitance Matrix C (pF)\n\n"
            f"```text\n"
            f"{C_matrix / 1e-12}\n"
            f"```\n"
        )

        # ----------------------------------------------------
        # Frequency chart
        # ----------------------------------------------------

        fig1, ax1 = plt.subplots(figsize=(9, 5))

        mode_names = [
            "Mode 1",
            "Mode 2",
            "Mode 3"
        ]

        ax1.bar(
            mode_names,
            frequencies / 1e6
        )

        ax1.set_title(
            "Natural Frequencies of Coupled LC Network"
        )

        ax1.set_ylabel(
            "Frequency (MHz)"
        )

        ax1.grid(
            axis="y",
            alpha=0.3
        )

        plt.tight_layout()

        # ----------------------------------------------------
        # Eigenvector chart
        # ----------------------------------------------------

        fig2, ax2 = plt.subplots(figsize=(9, 5))

        x = np.arange(3)
        width = 0.25

        ax2.bar(
            x - width,
            eigenvectors[:, 0],
            width,
            label="Mode 1"
        )

        ax2.bar(
            x,
            eigenvectors[:, 1],
            width,
            label="Mode 2"
        )

        ax2.bar(
            x + width,
            eigenvectors[:, 2],
            width,
            label="Mode 3"
        )

        ax2.axhline(0, linewidth=1)

        ax2.set_xticks(x)
        ax2.set_xticklabels([
            "Circuit 1",
            "Circuit 2",
            "Circuit 3"
        ])

        ax2.set_ylabel(
            "Normalized Current"
        )

        ax2.set_title(
            "Current Distribution in Each Natural Mode"
        )

        ax2.legend()
        ax2.grid(axis="y", alpha=0.3)

        plt.tight_layout()

        return (
            result_text,
            matrix_text,
            fig1,
            fig2
        )

    except Exception as e:

        return (
            f"❌ **Error:** {str(e)}",
            "",
            None,
            None
        )

In [5]:
# ============================================================
# CELL 19: BUILD FRONTEND
# ============================================================

import gradio as gr


with gr.Blocks(
    title="Electrical Network Natural Modes"
) as demo:

    gr.Markdown(
        """
        # ⚡ Electrical Network Natural Modes

        ### B.Tech Mathematics Mini Project — B9

        **Eigenvalues → Natural Frequencies**
        **Eigenvectors → Current Distribution**
        **Frequency Response → Resonance**

        Enter the circuit parameters below and click
        **Calculate Natural Modes**.
        """
    )

    # ========================================================
    # INPUT SECTION
    # ========================================================

    gr.Markdown("## 🔧 Circuit Parameters")

    with gr.Row():

        with gr.Column():

            gr.Markdown("### Inductors")

            L1 = gr.Number(
                value=100,
                label="L1 (nH)"
            )

            L2 = gr.Number(
                value=100,
                label="L2 (nH)"
            )

            L3 = gr.Number(
                value=100,
                label="L3 (nH)"
            )

        with gr.Column():

            gr.Markdown("### Mutual Inductance")

            M12 = gr.Number(
                value=20,
                label="M12 (nH)"
            )

            M13 = gr.Number(
                value=0,
                label="M13 (nH)"
            )

            M23 = gr.Number(
                value=20,
                label="M23 (nH)"
            )

        with gr.Column():

            gr.Markdown("### Capacitors")

            C1 = gr.Number(
                value=10,
                label="C1 (pF)"
            )

            C2 = gr.Number(
                value=10,
                label="C2 (pF)"
            )

            C3 = gr.Number(
                value=10,
                label="C3 (pF)"
            )

    calculate_button = gr.Button(
        "⚡ Calculate Natural Modes",
        variant="primary"
    )

    # ========================================================
    # OUTPUT SECTION
    # ========================================================

    gr.Markdown("## 📊 Results")

    result_output = gr.Markdown()

    matrix_output = gr.Markdown()

    with gr.Row():

        frequency_plot = gr.Plot(
            label="Natural Frequencies"
        )

        eigenvector_plot = gr.Plot(
            label="Eigenvectors"
        )

    # ========================================================
    # BUTTON CONNECTION
    # ========================================================

    calculate_button.click(
        fn=run_frontend,

        inputs=[
            L1, L2, L3,
            M12, M13, M23,
            C1, C2, C3
        ],

        outputs=[
            result_output,
            matrix_output,
            frequency_plot,
            eigenvector_plot
        ]
    )


# Launch application
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://73d4c76f119bf32549.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [13]:
# ============================================================
# CELL 20: PROJECT INFORMATION FRONTEND
# ============================================================

import gradio as gr

project_app = gr.Blocks()

with project_app:

    gr.Markdown("# Electrical Network Natural Modes")

    gr.Markdown("B.Tech Mathematics Mini Project - B9")

    gr.Markdown("## Mathematical Model")

    gr.Markdown(
        "L * d2i/dt2 + C^(-1) * i = 0"
    )

    gr.Markdown(
        "Eigenvalue equation: C^(-1) v = lambda L v"
    )

    gr.Markdown(
        "Natural frequency: f = sqrt(lambda) / (2*pi)"
    )

    gr.Markdown("## Project Results")

    gr.Markdown(
        "Eigenvalues represent the natural modes of the circuit."
    )

    gr.Markdown(
        "Eigenvectors represent the relative current distribution."
    )

    gr.Markdown(
        "Resonance occurs near the natural frequencies."
    )

    gr.Markdown("## Benchmark Circuit")

    gr.Markdown(
        "L1 = 100 nH"
    )

    gr.Markdown(
        "L2 = 100 nH"
    )

    gr.Markdown(
        "L3 = 100 nH"
    )

    gr.Markdown(
        "M12 = 20 nH"
    )

    gr.Markdown(
        "M13 = 0 nH"
    )

    gr.Markdown(
        "M23 = 20 nH"
    )

    gr.Markdown(
        "C1 = 10 pF"
    )

    gr.Markdown(
        "C2 = 10 pF"
    )

    gr.Markdown(
        "C3 = 10 pF"
    )

    gr.Markdown("## Applications")

    gr.Markdown(
        "RF filter design"
    )

    gr.Markdown(
        "Antenna tuning"
    )

    gr.Markdown(
        "Coupled resonators"
    )

    gr.Markdown(
        "Communication circuits"
    )

    gr.Markdown(
        "Microwave and RF systems"
    )

    gr.Markdown("## Conclusion")

    gr.Markdown(
        "Eigenvalues determine the natural frequencies."
    )

    gr.Markdown(
        "Eigenvectors determine the current distribution "
        "in each natural mode."
    )

    gr.Markdown(
        "The frequency response shows resonance near "
        "the natural frequencies."
    )


project_app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5afc1f276f7d6bb1ca.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
